In [1]:
from pymol import cmd
import py3Dmol
from openbabel import pybel

from rdkit import Chem
from rdkit.Chem import AllChem

import pandas as pd

import sys, os, random
sys.path.insert(1, './utilities')
import subprocess
from pathlib import Path
from tqdm import tqdm
from utils import getbox
import re

import warnings
warnings.filterwarnings('ignore')

In [19]:
#01 Menentukan direktori data
HERE = Path.cwd()
DATA = HERE / "data"

# Target protein pdb 
PDB = "5Y2O"

# Direktori Output
OUTPUT_DIR = DATA / PDB
DOCKED_DIR = OUTPUT_DIR / "docked"
LIGAND_DIR = OUTPUT_DIR / "ligand"

In [20]:
#02 Membuat direktori output apabila belum ada
os.makedirs(DOCKED_DIR, exist_ok=True)
os.makedirs(LIGAND_DIR, exist_ok=True)

In [10]:
#03 Memuat data dari hasil filtrasi substructure
ligands = pd.read_csv(
    DATA / "03NPstructureFiltered.csv",
    index_col=0,
)
ligands.head()

,pref_name,org_name,molecule_chembl_id,pubchem_cid,smiles,ro5_fulfilled,ROMol
0,Aceglumate,['Capsella bursa-pastoris'],CHEMBL2142890,185,CC(=O)NC(CCC(=O)O)C(=O)O,True,<rdkit.Chem.rdchem.Mol object at 0x7f5a1af03990>
1,Vitextrifolin B,['Vitex trifolia'],CHEMBL2391538,71579298,CO[C@H]1C[C@@]2(CC[C@@]3(O2)[C@H](C)CC[C@H]2C(...,True,<rdkit.Chem.rdchem.Mol object at 0x7f5a1af02e30>
2,Valine,"['Allium sativum', 'Pogostemon cablin', 'Solan...",CHEMBL43068,6971018;6287;88733505,CC(C)[C@H](N)C(=O)O,True,<rdkit.Chem.rdchem.Mol object at 0x7f5a1af03a00>
3,Leptorumol,['Pisonia aculeata'],CHEMBL1802146,56683359,Cc1c(O)c(C)c2occc(=O)c2c1O,True,<rdkit.Chem.rdchem.Mol object at 0x7f5a1af03bc0>
4,Debromoaplysiatoxin,['Cinnamomum camphora'],CHEMBL2148106,5352033,CO[C@@H](CC[C@H](C)[C@H]1O[C@@]23C[C@H](OC(=O)...,True,<rdkit.Chem.rdchem.Mol object at 0x7f5a1af03df0>


In [12]:
#04 Definisi path protein dan ligand yang telah dipisahkan
prot_pdb_file = OUTPUT_DIR / f"{PDB}_clean.pdb"
lig_mol2_file = OUTPUT_DIR / f"{PDB}_lig.mol2"
prot_pdb_h_file = OUTPUT_DIR / f"{PDB}_clean_H.pdb"

In [13]:
#05 Extract protein dan ligand dari PDB
cmd.fetch(code=PDB,type='pdb1')
cmd.select(name='Prot',selection='polymer.protein')
cmd.select(name='Lig',selection='organic')
cmd.save(filename=prot_pdb_file,format='pdb',selection='Prot')
cmd.save(filename=lig_mol2_file, format='mol2',selection='Lig')
cmd.delete('all')

In [ ]:
#06 Visualisasi protein dan ligand
view = py3Dmol.view()
view.removeAllModels()
view.setViewStyle({'style':'outline','color':'black','width':0.1})

view.addModel(open(prot_pdb_file,'r').read(),format='pdb')
Prot=view.getModel()
Prot.setStyle({'cartoon':{'arrows':True, 'tubes':True, 'style':'oval', 'color':'white'}})
view.addSurface(py3Dmol.VDW,{'opacity':0.6,'color':'white'})

view.addModel(open(lig_mol2_file,'r').read(),format='mol2')
ref_m = view.getModel()
ref_m.setStyle({},{'stick':{'colorscheme':'greenCarbon','radius':0.2}})

view.zoomTo()
view.show()

In [15]:
#07 Add hydrogens and protonate the protein
!./bin/lepro_linux_x86 {prot_pdb_file}
os.rename('pro.pdb', prot_pdb_h_file)

In [16]:
#08 Identifikasi docking box berdasarkan ligand kontrol
cmd.load(filename=str(prot_pdb_h_file), format='pdb', object='prot')
cmd.load(filename=str(lig_mol2_file), format='mol2', object='lig')

center, size = getbox(selection='lig', extending=6.0, software='vina')
cmd.delete('all')
print(center)
print(size)

{'center_x': -48.38199996948242, 'center_y': -1.3345000743865967, 'center_z': 77.3125}
{'size_x': 16.48999786376953, 'size_y': 21.11899995803833, 'size_z': 24.869003295898438}


In [17]:
#09 Molekular docking dengan SMINA untuk ligand kontrol

# Load and process the control ligand
mol = [m for m in pybel.readfile("mol2", str(lig_mol2_file))][0]
mol.addh()  # Add hydrogens
mol.write("mol2", str(lig_mol2_file), overwrite=True)

# Perform docking for the ligand
docked_output = DOCKED_DIR / "docked_ligand_control.sdf" 
log_file = DOCKED_DIR / "docking_ligand_control.log"

subprocess.run([
    "./bin/smina",
    "-r", str(prot_pdb_h_file),  
    "-l", str(lig_mol2_file),  
    "-o", str(docked_output),
    "--center_x", str(center['center_x']),  
    "--center_y", str(center['center_y']),
    "--center_z", str(center['center_z']),
    "--size_x", str(size['size_x']),  
    "--size_y", str(size['size_y']),
    "--size_z", str(size['size_z']),
    "--exhaustiveness", "8",
    "--num_modes", "5"
], stdout=open(log_file, 'w'), stderr=subprocess.STDOUT)

CompletedProcess(args=['./bin/smina', '-r', '/mnt/d/anagenic/workshop/data/5Y2O/5Y2O_clean_H.pdb', '-l', '/mnt/d/anagenic/workshop/data/5Y2O/5Y2O_lig.mol2', '-o', '/mnt/d/anagenic/workshop/data/5Y2O/docked/docked_ligand_control.sdf', '--center_x', '-48.38199996948242', '--center_y', '-1.3345000743865967', '--center_z', '77.3125', '--size_x', '16.48999786376953', '--size_y', '21.11899995803833', '--size_z', '24.869003295898438', '--exhaustiveness', '8', '--num_modes', '5'], returncode=0)

In [ ]:
#10 Molekular docking dengan SMINA untuk dataset ligand 

# Iterasi pada tiap ligand dalam dataset
for index, row in tqdm(ligands.iterrows(), total=len(ligands), desc="Processing ligands"):
    smiles = row['smiles']
    
    # Generate a molecule from SMILES
    mol = pybel.readstring("smi", smiles)
    mol.title = f"ligand_{index}"
    
    # Save each ligand to a mol2 file
    ligand_mol2_file = LIGAND_DIR / f"ligand_{index}.mol2"
    mol.make3D('mmff94s')
    mol.localopt(forcefield='mmff94s', steps=500)
    mol.write("mol2", str(ligand_mol2_file), overwrite=True)

    # Perform docking for the ligand
    docked_output = DOCKED_DIR / f"docked_ligand_{index}.sdf"
    log_file = DOCKED_DIR / f"docking_ligand_{index}.log"
    
    subprocess.run([
        "./bin/smina",
        "-r", str(prot_pdb_h_file),
        "-l", str(ligand_mol2_file),
        "-o", str(docked_output),
        "--center_x", str(center['center_x']),
        "--center_y", str(center['center_y']),
        "--center_z", str(center['center_z']),
        "--size_x", str(size['size_x']),
        "--size_y", str(size['size_y']),
        "--size_z", str(size['size_z']),
        "--exhaustiveness", "8",
        "--num_modes", "5"
    ], stdout=open(log_file, 'w'), stderr=subprocess.STDOUT) 

In [ ]:
#11 Membaca hasil docking dan tabulasi data menjadi datafg
data = []

# Function to parse SDF files and extract information
def parse_sdf(sdf_file, protein, sdf_filename):
    suppl = Chem.SDMolSupplier(sdf_file)
    if not suppl:
        return
    for index, mol in enumerate(suppl):
        if mol is None:
            continue
        props = mol.GetPropsAsDict()
        affinity = props.get('minimizedAffinity', 'N/A')
        
        data.append({
            'SDF Filename': sdf_filename,
            'Mode': index + 1,  # Assuming mode is the index of the pose + 1
            'Affinity (kcal/mol)': affinity
        })

# Loop through SDF files
for result_file in os.listdir(DOCKED_DIR):
    if result_file.endswith('.sdf'):
        sdf_file_path = os.path.join(DOCKED_DIR, result_file)
        sdf_filename = os.path.basename(sdf_file_path)
        parse_sdf(sdf_file_path, prot_pdb_h_file, sdf_filename)

# Create a DataFrame
df = pd.DataFrame(data)
df.head()



In [24]:
#12 Kombinasi hasil docking dengan list species dan NP
def get_species_and_pref_name(sdf_filename, df_2):
    match = re.search(r'docked_ligand_(\d+)', sdf_filename)
    
    if match:
        index = int(match.group(1))  # Extract the index from the SDF filename
        if index < len(df_2):
            species_list = df_2.loc[index, 'org_name']
            pref_name = df_2.loc[index, 'pref_name']
            return species_list, pref_name
    
    elif "control" in sdf_filename:
        return "Control species_list", "Control pref_name"
    
    return None, None  # Return None if no match is found

# Apply function to fill 'species_list' and 'pref_name' columns
df['species_list'], df['pref_name'] = zip(*df['SDF Filename'].apply(lambda x: get_species_and_pref_name(x, ligands)))
df.head()


In [26]:
#13 Menyimpan hasil analisis ke dalam file CSV
df.to_csv(DATA / f"04results{PDB}.csv")

In [ ]:
#14  Identifikasi hasil docking dengan ligand tertentu 

#Specify the SDF file you want to inspect
sdf_file_to_inspect = DOCKED_DIR / 'docked_ligand_11.sdf'

view = py3Dmol.view()
view.removeAllModels()
view.setViewStyle({'style':'outline','color':'black','width':0.1})

view.addModel(open(str(prot_pdb_h_file),'r').read(),format='pdb')
Prot=view.getModel()
Prot.setStyle({'cartoon':{'arrows':True, 'tubes':True, 'style':'oval', 'color':'white'}})
view.addSurface(py3Dmol.VDW,{'opacity':0.6,'color':'white'})


# Load and visualize the first pose of the ligand
poses = Chem.SDMolSupplier(sdf_file_to_inspect)
if poses is not None:
    first_pose = poses[0]
    if first_pose is not None:
        p = Chem.MolToMolBlock(first_pose)
        view.addModel(p, 'mol')
        z = view.getModel()
        z.setStyle({},{'stick':{'colorscheme':'greenCarbon','radius':0.2}})  # Use a fixed color for the ligand



view.zoomTo()
view.show()